# Website Scraper

This section creates a **Selenium-based web scraper**.

## Steps

1. Load the website using Selenium

2. Extract the HTML

3. Parse it using BeautifulSoup

4. Remove unnecessary elements

5. Extract clean text

### Architecture

Selenium → BeautifulSoup → Clean Text → LLM

In [7]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from openai import OpenAI
from IPython.display import Markdown, display

# ---------------------------------------------------------
# LLM CLIENTS
# ---------------------------------------------------------

openai_client = OpenAI()

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


# ---------------------------------------------------------
# SELENIUM
# ---------------------------------------------------------

def create_driver(headless=True):

    options = Options()

    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")

    return webdriver.Chrome(options=options)


# ---------------------------------------------------------
# WEBSITE SCRAPER
# ---------------------------------------------------------

def scrape_website(url, max_chars=20_000):

    driver = create_driver()

    try:
        driver.get(url)

        # Wait until page body loads
        WebDriverWait(driver, 10).until(
            lambda d: d.find_element("tag name", "body")
        )

        html = driver.page_source

    finally:
        driver.quit()

    soup = BeautifulSoup(html, "html.parser")

    # Get title
    title = soup.title.get_text(strip=True) if soup.title else ""

    # Remove unnecessary HTML
    unwanted_tags = [
        "script",
        "style",
        "svg",
        "img",
        "input",
        "button",
        "noscript",
        "iframe"
    ]

    for tag in soup(unwanted_tags):
        tag.decompose()

    # Prefer main/article content
    content = (
        soup.find("main")
        or soup.find("article")
        or soup.body
    )

    if content:
        text = content.get_text(
            separator="\n",
            strip=True
        )
    else:
        text = ""

    # Clean blank lines
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    text = "\n".join(lines)

    return {
        "url": url,
        "title": title,
        "text": text[:max_chars]
    }


# ---------------------------------------------------------
# PROMPTS
# ---------------------------------------------------------

system_prompt = """
You are an assistant that analyzes website content.

Use only the website content provided to you.
Do not invent information.

Provide a concise summary of the important information.
Use clear Markdown.
"""


def messages_for(website):

    user_prompt = f"""
Please summarize this website.

Title: {website["title"]}
URL: {website["url"]}

Website Content:

{website["text"]}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


# ---------------------------------------------------------
# SUMMARIZER
# ---------------------------------------------------------

def summarize(website, provider="ollama", model=None):

    if provider == "openai":

        client = openai_client

        if model is None:
            model = "gpt-4.1-mini"

    elif provider == "ollama":

        client = ollama_client

        if model is None:
            model = "gemma3"

    else:
        raise ValueError(
            "provider must be 'openai' or 'ollama'"
        )

    response = client.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )

    return response.choices[0].message.content


# ---------------------------------------------------------
# MAIN FUNCTION YOU CALL
# ---------------------------------------------------------

def scrape(url, provider=None, model=None, max_chars=20_000):

    website = scrape_website(
        url,
        max_chars=max_chars
    )

    # Scrape only
    if provider is None:
        return website

    # Scrape + summarize
    return summarize(
        website,
        provider=provider,
        model=model
    )

In [8]:
summary = scrape(
    "https://www.anthropic.com/news",
    provider="openai",
    model="gpt-4.1-mini"
)

display(Markdown(summary))

# Summary of Anthropic Newsroom

## Contact Information
- **Press inquiries:** press@anthropic.com
- **Non-media inquiries:** Support information available on site.
- **Media assets:** Press kit available for download.

## Recent Announcements & News Highlights
- **Sep 22, 2026:**  
  - *Claude Opus 5.5* introduced: Matches Claude Fable 5.1 performance on most tasks, while reducing operating costs by 40% compared to Opus 5.  
  - *The Situation Report:* Claude AI is being used by global health organizations to combat a rare Ebola strain spreading in the Democratic Republic of Congo.

- **Sep 1, 2026:**  
  - *Claude Fable 5.1 and Claude Mythos 5.1* launched as advanced models for coding, knowledge work, and scientific research support.

- **Sep 10, 2026:**  
  - *Detecting and countering misuse of AI:* Report on efforts by Anthropic’s Threat Intelligence team to disrupt malicious use of Claude AI, including case studies and evolution since 2025.

## Other Noteworthy Items
- **Sep 23, 2026:** Claude discovers a novel enzyme system with CRISPR-like repeats, demonstrating scientific research applications.
- Partnerships and programs launched, including collaborations with Accenture and the Life Sciences Verification Program.
- Ongoing efforts improving AI safety, alignment, security, biology safeguards, and monitoring AI impact on wellbeing.
- Introduction of new standards and technologies such as the Model Hardware Standard and Claude’s text watermark.

The newsroom focuses heavily on AI model advancements, safety/security updates, scientific contributions, and partnerships supporting AI evaluation and responsible use.

In [11]:
summary = scrape(

    "https://www.anthropic.com/news",

    provider="ollama"

)
display(Markdown(summary))

Here’s a summary of the Anthropic Newsroom website content:

**Key Announcements & Product Updates:**

*   **Claude Opus 5.5:** Launched September 22, 2026, Opus 5.5 offers performance comparable to Claude Fable 5.1 at 40% lower cost.
*   **Claude Fable 5.1 & Claude Mythos 5.1:** Introduced September 1, 2026, these models are focused on coding and knowledge work with research capabilities.
*   **Threat Intelligence Reports:** A report was published September 10, 2026 detailing efforts to detect and counter misuse of Claude by malicious actors.

**Recent News & Activities:**

*   **Scientific Discoveries:** September 23, 2026, Claude identified a novel enzyme system.
*   **Partnerships:** September 18, 2026, Anthropic is partnering with Accenture.
*   **Safety & Alignment Efforts:** Ongoing initiatives include developing enterprise safeguards, improving alignment and security, and expanding support for scientists. Anthropic is also implementing a text watermark for Claude.

In [10]:
summary = scrape(

    "https://www.anthropic.com/news",

    provider="ollama",

    model="gpt-oss"

)

display(Markdown(summary))

**Anthropic Newsroom (as of the provided snapshot)**  

| Section | Key Points |
|---------|------------|
| **Contact** | Press inquiries: **press@anthropic.com**; Non‑media inquiries and support links (not expanded here). |
| **Media Assets** | Press kit available for download. |
| **Recent Announcements** | • **Sep 22 2026** – *Claude Opus 5.5*: matches performance of Claude Fable 5.1, 40 % cheaper to run. <br>• **Sep 22 2026** – *The Situation Report*: Ebola outbreak in DRC highlighted, Claude deployed for rapid response. <br>• **Sep 1 2026** – *Claude Fable 5.1 & Claude Mythos 5.1*: top‑tier coding/knowledge models, early scientific insight. <br>• **Sep 10 2026** – *Detecting & Countering AI Misuse*: Threat Intelligence update with case studies and evolving attack patterns. |
| **News Highlights** | • **Sep 23 2026 – Science**: Claude discovers a novel enzyme system with CRISPR‑like repeats. <br>• **Sep 18 2026 – Announcements**: Partnering with Accenture for embedded evaluation. <br>• **Sep 17 2026 – Announcements**: Life Sciences Verification Program launch. <br>• **Sep 1 2026 – Announcements**: Enterprise Frontier Safeguards development. <br>• **Aug 31 2026 – Announcements**: Alignment & security improvements. <br>• **Aug 27 2026 – Announcements**: Model Hardware Standard preview; Support expansion for scientists. <br>• **Aug 25 2026 – Announcements**: Funding for AI‑well‑being impact studies. <br>• **Aug 14 2026 – Announcements**: Explanation of Claude’s text watermark. <br>• **Aug 7 2026 – Product**: Fable 5 biology safeguards enhancements. |

**Overview**
- The site functions as a press and communications hub, offering a press kit, contact email, and categorized news items.  
- Recent focus is on new Claude model releases, safety and misuse mitigation, and collaborations that leverage AI for scientific and business applications.  
- Detailed dates and titles allow quick navigation to the most recent updates.